In [1]:
import pandas as pd
from scipy.sparse import coo_matrix
import numpy as np

# Step 1: Load the data
rtngs = pd.read_csv('Ratings.csv', delimiter=';')
print(rtngs)

# Step 2: Data Cleaning
# Remove duplicates
filtrd_rtngs = rtngs.drop_duplicates(subset=['User-ID', 'ISBN'])
print("Remove duplicates")
print(filtrd_rtngs)

# Step 5: Create the Sparse Matrix
# Map user and book IDs to sequential indices
Usr_indx = {user: indx for indx, user in enumerate(filtrd_rtngs['User-ID'].unique())}
bk_indx = {book: indx for indx, book in enumerate(filtrd_rtngs['ISBN'].unique())}

# Convert user and book IDs to corresponding indices
filtrd_rtngs['Usr_indx'] = filtrd_rtngs['User-ID'].map(Usr_indx)
filtrd_rtngs['bk_indx'] = filtrd_rtngs['ISBN'].map(bk_indx)
print(filtrd_rtngs)

# Creating the sparse matrix
dta = filtrd_rtngs['Rating']
r__ow = filtrd_rtngs['Usr_indx']
c__ol = filtrd_rtngs['bk_indx']
sprse_mtrx = coo_matrix((dta, (r__ow, c__ol)), shape=(len(Usr_indx), len(bk_indx)))
print("Sparse matrix:")
print(sprse_mtrx)

from sklearn.datasets import dump_svmlight_file
F = sprse_mtrx
P = filtrd_rtngs['Usr_indx'].unique()

dump_svmlight_file(F, P, "User__BkRating.libsvm")

# Step 3: Calculate the Rating Proportion for Each User
# calculating the total number of ratings per user
ttl_rtngs_pr_usr = rtngs.groupby('User-ID').size()
print("calculating the total number of ratings per user")
print(ttl_rtngs_pr_usr)

# calculating the number of non-zero ratings per user
non__zero__rtngs_pr__usr = rtngs[rtngs['Rating'] != 0].groupby('User-ID').size()
print("calculating the number of non-zero ratings per user")
print(non__zero__rtngs_pr__usr)

# Calculating the rating proportion for each user
rtng_prpton__pr__usr = non__zero__rtngs_pr__usr / ttl_rtngs_pr_usr
print("Calculating the rating proportion for each user")
print(rtng_prpton__pr__usr)

# Step 4: Filtering the users based on the threshold
# Set the threshold (e.g., 0.3)
threshold = 0.3

# Filtering the users who meet the threshold
valid__usrs = rtng_prpton__pr__usr[rtng_prpton__pr__usr >= threshold].index
print("Filtering users who meet the threshold")
print(valid__usrs)

# Filtering the ratings data to include only valid users
flterd__rtngs = rtngs[rtngs['User-ID'].isin(valid__usrs)]
print("Filtering the ratings data to include only valid users")
print(flterd__rtngs)
print("How many records have zero value")
print(flterd__rtngs[flterd__rtngs['Rating'] == 0].shape[0])

# Manual saving of Libsvm format
# Step 6: Saving the record in libsvm Format
def save___libsvm_grped(sprse_mtrx, file_path):
    coo = sprse_mtrx.tocoo()
    with open(file_path, 'w') as f:
        crnt__usr = None
        line = ""
        for h, k, l in zip(coo.row, coo.col, coo.data):
            if h != crnt__usr:
                if crnt__usr is not None:
                    f.write(line + "\n")
                crnt__usr = h
                line = f"{crnt__usr}"
            line += f" {k+1}:{l}"
        if line:
            f.write(line + "\n")

save___libsvm_grped(sprse_mtrx, 'user_book_ratings.libsvm')

         User-ID         ISBN  Rating
0         276725   034545104X       0
1         276726   0155061224       5
2         276727   0446520802       0
3         276729   052165615X       3
4         276729   0521795028       6
...          ...          ...     ...
1149775   276704   1563526298       9
1149776   276706   0679447156       0
1149777   276709   0515107662      10
1149778   276721   0590442449      10
1149779   276723  05162443314       8

[1149780 rows x 3 columns]
Remove duplicates
         User-ID         ISBN  Rating
0         276725   034545104X       0
1         276726   0155061224       5
2         276727   0446520802       0
3         276729   052165615X       3
4         276729   0521795028       6
...          ...          ...     ...
1149775   276704   1563526298       9
1149776   276706   0679447156       0
1149777   276709   0515107662      10
1149778   276721   0590442449      10
1149779   276723  05162443314       8

[1149780 rows x 3 columns]
         User-

In [1]:
import pandas as pd
from scipy.sparse import coo_matrix
import numpy as np

# Step 1: Load the data from the CSV file
ratings_dataset = pd.read_csv("Ratings.csv", delimiter=';')
print(ratings_dataset)

# Step 2: Remove duplicate entries based on unique user and book combinations
distinct_ratings = ratings_dataset.drop_duplicates(subset=['User-ID', 'ISBN'])
print("Removed duplicate entries:")
print(distinct_ratings)

# Step 3: Map user and book IDs to sequential indices
# Create dictionaries for mapping unique user and book IDs to new indices
user_id_to_index_map = {user_id: index for index, user_id in enumerate(distinct_ratings['User-ID'].unique())}
book_id_to_index_map = {book_id: index for index, book_id in enumerate(distinct_ratings['ISBN'].unique())}

# Apply the mappings to create new index columns for users and books
distinct_ratings['user_index'] = distinct_ratings['User-ID'].map(user_id_to_index_map)
distinct_ratings['book_index'] = distinct_ratings['ISBN'].map(book_id_to_index_map)
print(distinct_ratings)

# Step 4: Build the sparse matrix for the ratings
# Specify the data and row/column indices for the sparse matrix
rating_values = distinct_ratings['Rating']
user_row_mapping = distinct_ratings['user_index']
book_column_mapping = distinct_ratings['book_index']
sparse_ratings_matrix = coo_matrix((rating_values, (user_row_mapping, book_column_mapping)), shape=(len(user_id_to_index_map), len(book_id_to_index_map)))
print("Generated sparse matrix:")
print(sparse_ratings_matrix)

# Step 5: Save the sparse matrix in libsvm format
from sklearn.datasets import dump_svmlight_file
matrix_data = sparse_ratings_matrix
unique_user_indices = distinct_ratings['user_index'].unique()

# Export the matrix and user indices to a libsvm format file
dump_svmlight_file(matrix_data, unique_user_indices, "UserBook_Ratings.libsvm")

# Step 6: Calculate the proportion of non-zero ratings per user
# Count the total number of ratings per user
total_ratings_count = ratings_dataset.groupby('User-ID').size()
print("Total number of ratings per user:")
print(total_ratings_count)

# Count the non-zero ratings per user
non_zero_ratings_count = ratings_dataset[ratings_dataset['Rating'] != 0].groupby('User-ID').size()
print("Non-zero ratings per user:")
print(non_zero_ratings_count)

# Calculate the ratio of non-zero ratings to total ratings for each user
non_zero_rating_ratio = non_zero_ratings_count / total_ratings_count
print("Non-zero rating ratio per user:")
print(non_zero_rating_ratio)

# Step 7: Filter users who meet a minimum rating ratio threshold
# Define the minimum threshold for valid users (e.g., 0.3)
rating_ratio_threshold = 0.3

# Identify users with a non-zero rating ratio above the threshold
filtered_user_ids = non_zero_rating_ratio[non_zero_rating_ratio >= rating_ratio_threshold].index
print("Users meeting the rating ratio threshold:")
print(filtered_user_ids)

# Filter the ratings data to include only entries from users meeting the threshold
filtered_ratings_data = ratings_dataset[ratings_dataset['User-ID'].isin(filtered_user_ids)]
print("Filtered ratings data for valid users:")
print(filtered_ratings_data)
print("Count of zero-value ratings in filtered data:")
print(filtered_ratings_data[filtered_ratings_data['Rating'] == 0].shape[0])

# Step 8: Manually save the sparse matrix in libsvm format for grouped users
def export_to_libsvm(sparse_matrix, output_file_path):
    coo_data = sparse_matrix.tocoo()
    with open(output_file_path, 'w') as file:
        current_user = None
        line_buffer = ""
        for row, col, value in zip(coo_data.row, coo_data.col, coo_data.data):
            if row != current_user:
                if current_user is not None:
                    file.write(line_buffer + "\n")
                current_user = row
                line_buffer = f"{current_user}"
            line_buffer += f" {col+1}:{value}"
        if line_buffer:
            file.write(line_buffer + "\n")

# Save the user-book ratings in libsvm format
export_to_libsvm(sparse_ratings_matrix, 'user_book_ratings_output.libsvm')

         User-ID         ISBN  Rating
0         276725   034545104X       0
1         276726   0155061224       5
2         276727   0446520802       0
3         276729   052165615X       3
4         276729   0521795028       6
...          ...          ...     ...
1149775   276704   1563526298       9
1149776   276706   0679447156       0
1149777   276709   0515107662      10
1149778   276721   0590442449      10
1149779   276723  05162443314       8

[1149780 rows x 3 columns]
Removed duplicate entries:
         User-ID         ISBN  Rating
0         276725   034545104X       0
1         276726   0155061224       5
2         276727   0446520802       0
3         276729   052165615X       3
4         276729   0521795028       6
...          ...          ...     ...
1149775   276704   1563526298       9
1149776   276706   0679447156       0
1149777   276709   0515107662      10
1149778   276721   0590442449      10
1149779   276723  05162443314       8

[1149780 rows x 3 columns]
     

In [3]:
import pandas as pd
from scipy.sparse import coo_matrix
from sklearn.datasets import dump_svmlight_file

# Step 1: Load the data from the CSV file
ratings_dataset = pd.read_csv("Ratings.csv", delimiter=';')
print(ratings_dataset)

# Step 2: Remove duplicate entries based on unique user and book combinations
distinct_ratings = ratings_dataset.drop_duplicates(subset=['User-ID', 'ISBN'])
print("Removed duplicate entries:")
print(distinct_ratings)

# Step 3: Map user and book IDs to sequential indices
# Create dictionaries for mapping unique user and book IDs to new indices
user_id_to_index_map = {user_id: index for index, user_id in enumerate(distinct_ratings['User-ID'].unique())}
book_id_to_index_map = {book_id: index for index, book_id in enumerate(distinct_ratings['ISBN'].unique())}

# Apply the mappings to create new index columns for users and books
distinct_ratings['user_index'] = distinct_ratings['User-ID'].map(user_id_to_index_map)
distinct_ratings['book_index'] = distinct_ratings['ISBN'].map(book_id_to_index_map)
print(distinct_ratings)

# Step 4: Build the sparse matrix for the ratings
rating_values = distinct_ratings['Rating']
user_row_mapping = distinct_ratings['user_index']
book_column_mapping = distinct_ratings['book_index']
sparse_ratings_matrix = coo_matrix((rating_values, (user_row_mapping, book_column_mapping)), 
                                   shape=(len(user_id_to_index_map), len(book_id_to_index_map)))
print("Generated sparse matrix:")
print(sparse_ratings_matrix)

# Step 5: Calculate the proportion of non-zero ratings per user
total_ratings_count = ratings_dataset.groupby('User-ID').size()
print("Total number of ratings per user:")
print(total_ratings_count)

non_zero_ratings_count = ratings_dataset[ratings_dataset['Rating'] != 0].groupby('User-ID').size()
print("Non-zero ratings per user:")
print(non_zero_ratings_count)

non_zero_rating_ratio = non_zero_ratings_count / total_ratings_count
print("Non-zero rating ratio per user:")
print(non_zero_rating_ratio)

# Step 6: Filter users who meet a minimum rating ratio threshold
rating_ratio_threshold = 0.3
filtered_user_ids = non_zero_rating_ratio[non_zero_rating_ratio >= rating_ratio_threshold].index
print("Users meeting the rating ratio threshold:")
print(filtered_user_ids)

filtered_ratings_data = ratings_dataset[ratings_dataset['User-ID'].isin(filtered_user_ids)]
print("Filtered ratings data for valid users:")
print(filtered_ratings_data)
print("Count of zero-value ratings in filtered data:")
print(filtered_ratings_data[filtered_ratings_data['Rating'] == 0].shape[0])

# Step 7: Save the sparse matrix in libsvm format without the first column
def export_to_libsvm_without_user(sparse_matrix, output_file_path):
    coo_data = sparse_matrix.tocoo()
    with open(output_file_path, 'w') as file:
        current_user = None
        line_buffer = ""
        for row, col, value in zip(coo_data.row, coo_data.col, coo_data.data):
            if row != current_user:
                if line_buffer:  # Write the buffer for the previous user
                    file.write(line_buffer.strip() + "\n")
                current_user = row
                line_buffer = ""  # Clear the buffer
            line_buffer += f" {col+1}:{value}"  # Add the book index and rating
        if line_buffer:  # Write the last user's data
            file.write(line_buffer.strip() + "\n")

# Save the user-book ratings in libsvm format
export_to_libsvm_without_user(sparse_ratings_matrix, 'user_book_ratings_output_cleaned.libsvm')

         User-ID         ISBN  Rating
0         276725   034545104X       0
1         276726   0155061224       5
2         276727   0446520802       0
3         276729   052165615X       3
4         276729   0521795028       6
...          ...          ...     ...
1149775   276704   1563526298       9
1149776   276706   0679447156       0
1149777   276709   0515107662      10
1149778   276721   0590442449      10
1149779   276723  05162443314       8

[1149780 rows x 3 columns]
Removed duplicate entries:
         User-ID         ISBN  Rating
0         276725   034545104X       0
1         276726   0155061224       5
2         276727   0446520802       0
3         276729   052165615X       3
4         276729   0521795028       6
...          ...          ...     ...
1149775   276704   1563526298       9
1149776   276706   0679447156       0
1149777   276709   0515107662      10
1149778   276721   0590442449      10
1149779   276723  05162443314       8

[1149780 rows x 3 columns]
     

In [1]:
import pandas as pd
from scipy.sparse import coo_matrix
from sklearn.datasets import dump_svmlight_file

# Step 1: Load the data from the CSV file
rtngs_dtst = pd.read_csv("Ratings.csv", delimiter=';')
print(rtngs_dtst)

# Step 2: Remove duplicate entries based on unique user and book combinations
dstnct_rtngs = rtngs_dtst.drop_duplicates(subset=['User-ID', 'ISBN'])
print("Removed duplicate entries:")
print(dstnct_rtngs)

# Step 3: Map user and book IDs to sequential indices
# Create dictionaries for mapping unique user and book IDs to new indices
usr__id__to__mp = {user_id: index for index, user_id in enumerate(dstnct_rtngs['User-ID'].unique())}
bkkk__id__to__mp = {book_id: index for index, book_id in enumerate(dstnct_rtngs['ISBN'].unique())}

# Apply the mappings to create new index columns for users and books
dstnct_rtngs['user_index'] = dstnct_rtngs['User-ID'].map(usr__id__to__mp)
dstnct_rtngs['book_index'] = dstnct_rtngs['ISBN'].map(bkkk__id__to__mp)
print(dstnct_rtngs)

# Step 4: Build the sparse matrix for the ratings
rtngs_vls = dstnct_rtngs['Rating']
usr_rw_mpng = dstnct_rtngs['user_index']
bkkk_clmn_mmpng = dstnct_rtngs['book_index']
srse_rtngs__mtx = coo_matrix((rtngs_vls, (usr_rw_mpng, bkkk_clmn_mmpng)), 
                             shape=(len(usr__id__to__mp), len(bkkk__id__to__mp)))
print("Generated sparse matrix:")
print(srse_rtngs__mtx)

# Step 5: Calculate the proportion of non-zero ratings per user
ttl_rtngs__cnt = rtngs_dtst.groupby('User-ID').size()
print("Total number of ratings per user:")
print(ttl_rtngs__cnt)

non_zero__rtngs__cnt = rtngs_dtst[rtngs_dtst['Rating'] != 0].groupby('User-ID').size()
print("Non-zero ratings per user:")
print(non_zero__rtngs__cnt)

non_zero__rtngs__rtio = non_zero__rtngs__cnt / ttl_rtngs__cnt
print("Non-zero rating ratio per user:")
print(non_zero__rtngs__rtio)

# Step 6: Filter users who meet a minimum rating ratio threshold
rtngs_rtio__trshld = 0.3
fltrd__usr__ids = non_zero__rtngs__rtio[non_zero__rtngs__rtio >= rtngs_rtio__trshld].index
print("Users meeting the rating ratio threshold:")
print(fltrd__usr__ids)

fltrd__rtngs__dta = rtngs_dtst[rtngs_dtst['User-ID'].isin(fltrd__usr__ids)]
print("Filtered ratings data for valid users:")
print(fltrd__rtngs__dta)
print("Count of zero-value ratings in filtered data:")
print(fltrd__rtngs__dta[fltrd__rtngs__dta['Rating'] == 0].shape[0])

# Step 7: Save the sparse matrix in libsvm format without the first column
def export_to_libsvm_without_user(srse_rtngs__mtx, output_file_path):
    coo_data = srse_rtngs__mtx.tocoo()
    with open(output_file_path, 'w') as file:
        crnt__usr = None
        ln__bffr = ""
        for rw, cl, vll in zip(coo_data.row, coo_data.col, coo_data.data):
            if rw != crnt__usr:
                if ln__bffr:  # Write the buffer for the previous user
                    file.write(ln__bffr.strip() + "\n")
                crnt__usr = rw
                ln__bffr = ""  # Clear the buffer
            ln__bffr += f" {cl+1}:{vll}"  # Add the book index and rating
        if ln__bffr:  # Write the last user's data
            file.write(ln__bffr.strip() + "\n")

# Save the user-book ratings in libsvm format
export_to_libsvm_without_user(srse_rtngs__mtx, 'user_book_ratings_output_cleanedddddddddddddf.libsvm')

         User-ID         ISBN  Rating
0         276725   034545104X       0
1         276726   0155061224       5
2         276727   0446520802       0
3         276729   052165615X       3
4         276729   0521795028       6
...          ...          ...     ...
1149775   276704   1563526298       9
1149776   276706   0679447156       0
1149777   276709   0515107662      10
1149778   276721   0590442449      10
1149779   276723  05162443314       8

[1149780 rows x 3 columns]
Removed duplicate entries:
         User-ID         ISBN  Rating
0         276725   034545104X       0
1         276726   0155061224       5
2         276727   0446520802       0
3         276729   052165615X       3
4         276729   0521795028       6
...          ...          ...     ...
1149775   276704   1563526298       9
1149776   276706   0679447156       0
1149777   276709   0515107662      10
1149778   276721   0590442449      10
1149779   276723  05162443314       8

[1149780 rows x 3 columns]
     